# 14 Native slices (cache v3) and relative-enhancement input (GPU)
Same machinery as `11_sweep`. Every run: DINOv2 ViT-B/14, top 2 blocks trained, dropout 0.5, all cohorts.
Two questions (see `docs/ideas.md`, items 6 and 13):
1. **Does resampling between slices matter?** v2 = 2 mm slices, v3 = native slices (both 0.7 mm in-plane).
2. **Does a scanner-independent input help?** `enhancement` = early, (early - pre)/pre, (late - pre)/pre.

Runs (each with seeds 0 and 1); the **first entry is the baseline**:
- `v2_early`: cache v2, early phase only (the current best setting);
- `v3_early`: cache v3, early phase only -> effect of native slices;
- `v2_rgb`: cache v2, raw pre/early/late -> effect of using all three phases;
- `v2_enh`: cache v2, relative-enhancement maps -> effect of the normalisation, compared with `v2_rgb`;
- `v3_enh`: cache v3, relative-enhancement maps.

1. Session options: **GPU**, Internet **ON**, secret `GITHUB_TOKEN`.
2. **+ Add Input**: **both** `mnj-cache-v2` and `mnj-cache-v3` (built with `01_build_cache`, `Z_MM = 0`).
   Each run is pointed at its cache version; the attached versions are printed first.
3. `DEBUG = True` first, then `DEBUG = False` and **Save Version -> Save & Run All (Commit)** (about 5.5 h).

Read the **seed-averaged** table: it averages each setting's two seeds before the paired comparison, so it
is less noisy than the single-seed rows.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "d1.11"        # code version; A posts the tag to use
PERSON = ""             # set to "A", "B" or "C"
CONFIG = "b3_vit"
SEEDS = [0, 1]
P = ["optim.patience=15", "model.backbone=facebook/dinov2-base", "model.train_blocks=2", "model.dropout=0.5"]
RUNS = {                          # name: {"cache": builder version, "set": overrides}; FIRST = baseline
    "v2_early": {"cache": 2, "set": P + ["model.input_phases=early"]},
    "v3_early": {"cache": 3, "set": P + ["model.input_phases=early"]},
    "v2_rgb":   {"cache": 2, "set": P + ["model.input_phases=rgb"]},
    "v2_enh":   {"cache": 2, "set": P + ["model.input_phases=enhancement"]},
    "v3_enh":   {"cache": 3, "set": P + ["model.input_phases=enhancement"]},
}
BASELINE = None                   # None = the first entry of RUNS, same seed
CACHE_VERSION = 2                 # used for entries without a "cache" key
DEBUG = False
TIME_BUDGET_H = 11.0              # stop starting new runs after this (Kaggle limit: 12 h)
KEEP_CHECKPOINTS = False          # 10 runs x ~350 MB; set True only if you will reuse the weights
PUSH_RESULTS = True               # commit results/<PERSON>/<sweep id>/ (small files only)

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
assert PERSON in ("A", "B", "C"), 'set PERSON = "A", "B" or "C" in the parameters cell'
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import glob, time
import pandas as pd

SEED = "sweep"   # used in the push commit message
SWEEP_ID = f"sweep_{CONFIG}_{time.strftime('%m%d-%H%M')}" + ("_debug" if DEBUG else "")
RES = f"/kaggle/working/results/{PERSON}/{SWEEP_ID}"
os.makedirs(RES, exist_ok=True)
json.dump({"config": CONFIG, "seeds": SEEDS, "runs": RUNS, "git_tag": GIT_TAG, "debug": DEBUG},
          open(f"{RES}/sweep.json", "w"), indent=2)
rows, t_start, durations = [], time.time(), []

def run_spec(spec):
    """A RUNS entry: a list of overrides, or {"set": [overrides], "cache": builder version}."""
    return (spec, CACHE_VERSION) if isinstance(spec, list) else (spec.get("set", []), spec.get("cache", CACHE_VERSION))

CACHES = {}   # {builder version: {cohort: dir}} of the attached caches; each run is pointed at one version
for depth in range(6):
    for f in glob.glob(os.path.join("/kaggle/input", *["*"] * depth, "cache_meta.json")):
        m = json.load(open(f))
        CACHES.setdefault(m.get("builder_version", 1), {})[m["cohort"]] = os.path.dirname(f)
print("attached caches:", {f"v{v}": sorted(d) for v, d in sorted(CACHES.items())})

def finished_run_dir(before):
    new = sorted(set(glob.glob("/kaggle/working/runs/*/")) - before, key=os.path.getmtime)
    return os.path.normpath(new[-1]) if new else None

for name, spec in RUNS.items():
    extra, version = run_spec(spec)
    for seed in SEEDS:
        used_h = (time.time() - t_start) / 3600
        est_h = (sum(durations) / len(durations) / 3600) if durations else 0
        if used_h + est_h > TIME_BUDGET_H:
            print(f"skip {name} seed {seed}: {used_h:.1f} h used, next run needs ~{est_h:.1f} h")
            rows.append({"name": name, "seed": seed, "status": "skipped (time)"})
            continue
        cmd = [sys.executable, "-m", "src.train", "--config", CONFIG, "--seed", str(seed), "--mode", "train",
               "--cache-root", "/kaggle/input", "--out", "/kaggle/working/runs", "--results-dir", RES]
        if version is not None:
            cmd += ["--cache-version", str(version)]
            if len(CACHES) > 1 and version in CACHES:   # several versions attached: name this one's folders
                cmd += ["--cache", *[f"{c}={d}" for c, d in CACHES[version].items()]]
        if DEBUG:
            cmd.append("--debug")
        if extra:
            cmd += ["--set", *extra]
        print(f"\n===== {name} seed {seed} (cache v{version}): {' '.join(extra) or '(defaults)'}", flush=True)
        before, t = set(glob.glob("/kaggle/working/runs/*/")), time.time()
        rc = subprocess.run(cmd).returncode
        durations.append(time.time() - t)
        run_dir = finished_run_dir(before)
        row = {"name": name, "seed": seed, "cache": version, "overrides": " ".join(extra),
               "status": "ok" if rc == 0 else f"exit {rc}",
               "run_id": os.path.basename(run_dir) if run_dir else None, "minutes": round(durations[-1] / 60, 1)}
        if rc == 0 and run_dir and os.path.exists(f"{run_dir}/metrics.json"):
            m = json.load(open(f"{run_dir}/metrics.json"))
            v = m["val"]
            log = pd.read_csv(f"{run_dir}/log.csv")
            row.update(val_auc=round(v["auc"], 4), ci_lo=round(v["auc_ci"][0], 3), ci_hi=round(v["auc_ci"][1], 3),
                       auprc=round(v["auprc"], 3), ece=round(v["ece"], 3), best_epoch=m["best_epoch"],
                       epochs_run=m["epochs_run"], max_val_auc_any_epoch=round(log.val_auc.max(), 4),
                       **{f"auc_{c}": round(x["auc"], 3) for c, x in v.get("by_cohort", {}).items()})
            print(f"===== {name} seed {seed}: val AUC {v['auc']:.3f} (CI {v['auc_ci'][0]:.3f}-{v['auc_ci'][1]:.3f}), "
                  f"best epoch {m['best_epoch']} of {m['epochs_run']}, {row['minutes']} min", flush=True)
        if run_dir and not KEEP_CHECKPOINTS:
            for f in glob.glob(f"{run_dir}/*.pt"):
                os.remove(f)
        rows.append(row)
        pd.DataFrame(rows).to_csv(f"{RES}/summary.csv", index=False)   # saved after every run
print(f"\nsweep done in {(time.time() - t_start) / 3600:.2f} h -> {RES}/summary.csv")

In [ ]:
from src import stats
from IPython.display import display
import matplotlib.pyplot as plt

S = pd.DataFrame(rows)
ok = S[S.status == "ok"].copy()
preds = {(r.name, r.seed): pd.read_csv(f"{RES}/{r.run_id}/val_preds.csv").set_index("pid") for r in ok.itertuples()}
if BASELINE:   # an earlier run (in the repo clone) is the baseline; every run here is compared with it
    base = BASELINE["name"]
    ext = pd.read_csv(BASELINE["val_preds"]).set_index("pid")
    print(f"baseline '{base}' from {BASELINE['val_preds']}: val AUC {stats.auc(ext.label, ext.prob):.4f}")
    get_base = lambda seed: ext
else:          # the first entry of RUNS, same seed
    base = next(iter(RUNS))
    get_base = lambda seed: preds.get((base, seed))
for col in ("vs_base", "vs_base_ci", "vs_base_p"):
    ok[col] = None
for i, r in ok.iterrows():   # paired bootstrap against the baseline
    b = get_base(r.seed)
    if b is None or r["name"] == base:
        continue
    p = preds[(r["name"], r.seed)]
    common = b.index.intersection(p.index)          # same val patients in both (all 177 normally)
    if len(common) < len(b):
        print(f"{r['name']}: compared on {len(common)} of {len(b)} baseline patients "
              f"(baseline AUC on these: {stats.auc(b.loc[common, 'label'], b.loc[common, 'prob']):.4f})")
    d = stats.paired_bootstrap_auc_diff(b.loc[common, "label"], b.loc[common, "prob"], p.loc[common, "prob"])
    ok.at[i, "vs_base"], ok.at[i, "vs_base_ci"], ok.at[i, "vs_base_p"] = (
        round(d["diff"], 3), f"{d['ci'][0]:+.3f} to {d['ci'][1]:+.3f}", round(d["p"], 3))
cols = ["name", "seed", "val_auc", "ci_lo", "ci_hi", "vs_base", "vs_base_ci", "vs_base_p", "auprc", "ece",
        "best_epoch", "epochs_run", "max_val_auc_any_epoch", "auc_duke", "auc_ispy1", "auc_ispy2", "minutes"]
print(f"baseline = '{base}'. vs_base = AUC(run) - AUC(baseline), paired bootstrap on the same val patients.")
display(ok[[c for c in cols if c in ok]].sort_values("val_auc", ascending=False))
if len(SEEDS) > 1:
    print("mean over seeds:")
    display(ok.groupby("name")[["val_auc", "auprc", "ece", "best_epoch"]].agg(["mean", "std"]).round(3))
    # each setting's seeds averaged into one prediction per patient (less seed noise), then compared
    # with the baseline's seed average on the same patients
    avg = {}
    for n in ok["name"].unique():
        ps = [preds[(n, s)] for s in ok[ok["name"] == n].seed]
        common = ps[0].index
        for q in ps[1:]:
            common = common.intersection(q.index)
        avg[n] = pd.DataFrame({"label": ps[0].loc[common, "label"],
                               "prob": sum(q.loc[common, "prob"] for q in ps) / len(ps)})
    b = ext if BASELINE else avg.get(base)
    if b is not None:
        rows_avg = []
        for n, p in avg.items():
            common = b.index.intersection(p.index)
            d = stats.paired_bootstrap_auc_diff(b.loc[common, "label"], b.loc[common, "prob"], p.loc[common, "prob"])
            rows_avg.append({"name": n, "seeds": int((ok["name"] == n).sum()),
                             "auc_seed_avg": round(stats.auc(p.label, p.prob), 4), "vs_base": round(d["diff"], 3),
                             "vs_base_ci": f"{d['ci'][0]:+.3f} to {d['ci'][1]:+.3f}", "vs_base_p": round(d["p"], 3)})
        print(f"seed-averaged predictions, paired vs '{base}':")
        A = pd.DataFrame(rows_avg)
        display(A)
        A.to_csv(f"{RES}/summary_seed_avg.csv", index=False)
if (S.status != "ok").any():
    display(S[S.status != "ok"][["name", "seed", "status"]])
ok.to_csv(f"{RES}/summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for r in ok.itertuples():
    log = pd.read_csv(f"{RES}/{r.run_id}/log.csv")
    axes[0].plot(log.epoch, log.val_auc, label=f"{r.name} s{r.seed}")
    axes[1].plot(log.epoch, log.train_loss, label=f"{r.name} s{r.seed}")
axes[0].set(title="val AUC per epoch", xlabel="epoch"); axes[1].set(title="train loss per epoch", xlabel="epoch")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RES}/curves.png", dpi=90); plt.show()

In [ ]:
# ==== PUSH the small result files to GitHub (results/<PERSON>/<run_id>/) ====
if PUSH_RESULTS and not DEBUG:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    # the clone holds only the tag, so fetch main into origin/main explicitly; on a rejected push
    # (someone else pushed first) rebuild the commit on the new main instead of rebasing a shallow clone
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", "+refs/heads/main:refs/remotes/origin/main")
        git("checkout", "--quiet", "-f", "-B", "main", "origin/main")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {CONFIG} seed {SEED} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", "main"); print("pushed"); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output and send it to A")